# BTL1 — Biểu diễn ảnh màu, lọc tín hiệu và biến đổi hình học

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

np.random.seed(0)

def show_row(imgs, titles, size=4):
    """Hiển thị nhiều ảnh trên 1 hàng. Ảnh 2D vẽ thang xám 0–255, ảnh 3D vẽ RGB."""
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size))
    for ax, im, t in zip(np.atleast_1d(axes), imgs, titles):
        if im.ndim == 2:
            ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(np.clip(im, 0, 255).astype(np.uint8))
        ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show()

### Load ảnh

In [ ]:
images = {
    'astronaut': data.astronaut(),
    'coffee':    data.coffee(),
    'cat':       data.chelsea(),
}
show_row(list(images.values()), list(images.keys()))

## 1. Biểu diễn ảnh màu và ảnh xám

### Màu → xám và xám → màu

In [ ]:
def rgb2gray(img):
    """RGB (H, W, 3) -> xám (H, W) theo Y = 0.299R + 0.587G + 0.114B."""
    w = np.array([0.299, 0.587, 0.114])
    return np.round(img[..., :3] @ w).astype(np.uint8)

def gray2rgb(gray):
    """Xám (H, W) -> (H, W, 3) bằng cách chép Y vào cả 3 kênh; không khôi phục được màu."""
    return np.dstack([gray, gray, gray])

img  = images['astronaut']
gray = rgb2gray(img)
back = gray2rgb(gray)
print('Gốc:', img.shape, '| Xám:', gray.shape, '| Xám -> 3 kênh:', back.shape)
print('Đỏ (255,0,0), lục (0,130,0) ->', rgb2gray(np.array([[[255, 0, 0], [0, 130, 0]]], dtype=np.uint8))[0])
show_row([img, gray, back], ['Gốc (RGB)', 'Xám', 'Xám -> RGB'])

### So sánh: trung bình cộng và có trọng số

In [ ]:
names  = ['đỏ', 'lục', 'lam', 'vàng', 'lục lam', 'hồng tím']
colors = np.array([[255, 0, 0], [0, 255, 0], [0, 0, 255],
                   [255, 255, 0], [0, 255, 255], [255, 0, 255]], dtype=np.uint8)
strip = np.repeat(np.repeat(colors[None], 60, axis=0), 60, axis=1)
avg = strip.mean(axis=2)
lum = rgb2gray(strip)
for n, a, l in zip(names, avg[0, ::60], lum[0, ::60]):
    print(f'{n:9s} trung bình {a:4.0f} | có trọng số {l:4d}')
show_row([strip, avg, lum], ['6 màu thuần', 'Trung bình (R+G+B)/3', 'Có trọng số'], size=5)

img_avg = img.mean(axis=2)
diff = np.abs(img_avg - gray)
print(f'Chênh lệch trên astronaut: trung bình {diff.mean():.1f}, lớn nhất {diff.max():.0f}')
show_row([img, img_avg, gray, diff / diff.max() * 255],
         ['Gốc', 'Trung bình (R+G+B)/3', 'Có trọng số', 'Chênh lệch giữa 2 ảnh xám'])

### Tách kênh màu (R, G, B)

In [ ]:
R, G, B = img[:, :, 0], img[:, :, 1], img[:, :, 2]
show_row([R, G, B], ['Kênh R', 'Kênh G', 'Kênh B'])

def keep_channel(img, c):
    """Giữ kênh c, đặt 2 kênh còn lại = 0 để xem kênh đó bằng đúng màu của nó."""
    out = np.zeros_like(img)
    out[:, :, c] = img[:, :, c]
    return out

show_row([keep_channel(img, c) for c in range(3)], ['Chỉ R', 'Chỉ G', 'Chỉ B'])

### Ghép kênh: tái tạo và tạo ảnh màu mới

In [ ]:
recon   = np.dstack([R, G, B])
swap_rb = np.dstack([B, G, R])
no_red  = np.dstack([np.zeros_like(R), G, B])
mix     = np.dstack([G, B, R])

assert np.array_equal(recon, img), 'ghép (R, G, B) phải trùng ảnh gốc'
print('OK: ghép (R, G, B) trùng khớp ảnh gốc')
show_row([recon, swap_rb, no_red, mix],
         ['Tái tạo (R,G,B)', 'Hoán đổi (B,G,R)', 'Thiếu đỏ (0,G,B)', 'Kết hợp (G,B,R)'])

## 2. Lọc ảnh: Low-pass & High-pass

### Convolution

In [ ]:
def conv2d(img, kernel):
    """Convolution 2D bằng numpy, ảnh ra cùng kích thước ảnh vào.

    Viền được đệm kiểu phản chiếu. Thay vì lặp từng pixel, lặp từng ô kernel:
    mỗi lần cộng cả ảnh dịch đi (i, j) nhân với trọng số k[i, j].
    """
    img = img.astype(np.float64)
    k = kernel[::-1, ::-1]
    kh, kw = k.shape
    H, W = img.shape
    p = np.pad(img, ((kh // 2, kh // 2), (kw // 2, kw // 2)), mode='reflect')
    out = np.zeros((H, W))
    for i in range(kh):
        for j in range(kw):
            out += k[i, j] * p[i:i + H, j:j + W]
    return out

x = np.random.rand(15, 20)
delta = np.zeros((3, 3)); delta[1, 1] = 1
shift = np.zeros((3, 3)); shift[1, 2] = 1
assert np.allclose(conv2d(x, delta), x), 'kernel delta phải giữ nguyên ảnh'
assert np.allclose(conv2d(x, shift)[:, 1:], x[:, :-1]), 'delta lệch phải phải dịch ảnh sang phải 1 pixel'
assert np.allclose(conv2d(np.full((8, 8), 50.0), np.ones((3, 3)) / 9), 50), 'ảnh phẳng qua mean phải giữ nguyên'
print('OK: conv2d đúng')

def to_display(x):
    """Lấy |x| rồi co về 0–255 theo phân vị 99 để hiển thị kết quả high-pass."""
    x = np.abs(x)
    return np.clip(x / (np.percentile(x, 99) + 1e-8) * 255, 0, 255)

gray_imgs = {name: rgb2gray(im) for name, im in images.items()}

### Low-pass: Mean và Gaussian

In [ ]:
def gaussian_kernel(sigma):
    """Kernel Gaussian 2D kích thước (2*ceil(3σ)+1)², chuẩn hoá tổng = 1."""
    r = int(np.ceil(3 * sigma))
    ax = np.arange(-r, r + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return k / k.sum()

mean_k  = np.ones((5, 5)) / 25
gauss_k = gaussian_kernel(1.5)
print('Mean 5x5, hàng giữa      :', mean_k[2])
print('Gaussian σ=1.5, hàng giữa:', np.round(gauss_k[len(gauss_k) // 2], 3))

for name, g in gray_imgs.items():
    show_row([g, conv2d(g, mean_k), conv2d(g, gauss_k)],
             [f'{name}: gốc', 'Mean 5x5', 'Gaussian σ=1.5'])

### Ảnh hưởng của tham số σ

In [ ]:
g = gray_imgs['astronaut']
sigmas = [1, 2, 4]
show_row([g] + [conv2d(g, gaussian_kernel(s)) for s in sigmas],
         ['Gốc'] + [f'Gaussian σ={s}' for s in sigmas])

### Low-pass để khử nhiễu

In [ ]:
def psnr(clean, test):
    """PSNR (dB) giữa ảnh sạch và ảnh cần đánh giá; càng cao càng giống ảnh sạch."""
    mse = np.mean((clean.astype(np.float64) - test) ** 2)
    return 10 * np.log10(255 ** 2 / mse)

g = gray_imgs['coffee']
noisy     = np.clip(g + np.random.normal(0, 25, g.shape), 0, 255)
den_mean  = conv2d(noisy, mean_k)
den_gauss = conv2d(noisy, gauss_k)

print(f'PSNR — có nhiễu: {psnr(g, noisy):.1f} dB | sau mean: {psnr(g, den_mean):.1f} dB'
      f' | sau Gaussian: {psnr(g, den_gauss):.1f} dB')
for s in [1, 2, 4]:
    print(f'Gaussian σ={s}: PSNR {psnr(g, conv2d(noisy, gaussian_kernel(s))):.1f} dB')
show_row([g, noisy, den_mean, den_gauss], ['Gốc', 'Có nhiễu (σ=25)', 'Sau mean 5x5', 'Sau Gaussian σ=1.5'])

### High-pass: Sobel và Laplacian

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
sobel_y = sobel_x.T
laplace = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=float)

for name, g in gray_imgs.items():
    gx, gy = conv2d(g, sobel_x), conv2d(g, sobel_y)
    mag = np.sqrt(gx**2 + gy**2)
    show_row([g, to_display(gx), to_display(gy), to_display(mag), to_display(conv2d(g, laplace))],
             [f'{name}: gốc', 'Sobel x', 'Sobel y', 'Sobel |∇|', 'Laplacian'], size=3.5)

### High-pass trên ảnh nhiễu

In [ ]:
show_row([noisy, to_display(conv2d(noisy, laplace)), to_display(conv2d(den_gauss, laplace))],
         ['Có nhiễu', 'Laplacian trực tiếp', 'Gaussian rồi Laplacian'])

### Làm nét (unsharp masking)

In [ ]:
g = gray_imgs['cat']
detail = g - conv2d(g, gauss_k)
sharp  = np.clip(g + detail, 0, 255)
show_row([g, to_display(detail), sharp], ['Gốc', 'Chi tiết = gốc − Gaussian', 'Làm nét = gốc + chi tiết'])

## 3. Biến đổi hình học

### Load ảnh

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import data

img = data.rocket()
plt.imshow(img)
plt.axis("off");
print(img.shape)

### Bilinear Interpolation

In [ ]:
def bilinear_interpolation(img, x, y):
    """
    Nội suy giá trị pixel tại tọa độ thực (x, y) trên ảnh img.
    """
    h, w = img.shape[:2]
    # Tạo mask lọc các điểm nằm ngoài biên ảnh gốc
    valid = (x >= 0) & (x < w - 1) & (y >= 0) & (y < h - 1)

    x0 = np.floor(x).astype(int)
    x1 = x0 + 1
    y0 = np.floor(y).astype(int)
    y1 = y0 + 1

    x0_c = np.clip(x0, 0, w - 1)
    x1_c = np.clip(x1, 0, w - 1)
    y0_c = np.clip(y0, 0, h - 1)
    y1_c = np.clip(y1, 0, h - 1)

    # Tính các trọng số
    wa = (x1 - x) * (y1 - y)
    wb = (x1 - x) * (y - y0)
    wc = (x - x0) * (y1 - y)
    wd = (x - x0) * (y - y0)

    if img.ndim == 3:
        wa = wa[..., np.newaxis]
        wb = wb[..., np.newaxis]
        wc = wc[..., np.newaxis]
        wd = wd[..., np.newaxis]
        valid = valid[..., np.newaxis]

    # Kết quả thu được là giá trị màu
    out = (wa * img[y0_c, x0_c] +
           wb * img[y1_c, x0_c] +
           wc * img[y0_c, x1_c] +
           wd * img[y1_c, x1_c])

    return np.where(valid, out, 0).astype(np.uint8)

### Inverse Mapping

In [ ]:
def warp_perspective(img, M, output_shape):
    """
    Ánh xạ ngược: Với mỗi tọa độ (xd, yd) trên ảnh kết quả,
    tìm tọa độ gốc: (xs, ys) = M^{-1} @ (xd, yd, 1)
    """
    out_h, out_w = output_shape
    M_inv = np.linalg.inv(M)

    # Tạo lưới tọa độ đích
    yd, xd = np.indices((out_h, out_w))
    coords_d = np.stack([xd.ravel(), yd.ravel(), np.ones(out_h * out_w)], axis=0) # 3 x N

    # Chiếu ngược về ảnh gốc
    coords_s = M_inv @ coords_d
    xs = coords_s[0] / coords_s[2]
    ys = coords_s[1] / coords_s[2]

    xs = xs.reshape(out_h, out_w)
    ys = ys.reshape(out_h, out_w)

    return bilinear_interpolation(img, xs, ys)


### Ma trận biến đổi Affine

In [ ]:
def affine_matrix(points_s, points_d):
    """
    Tính ma trận Affine 3x3 từ 3 cặp điểm tương ứng: points_d = M @ points_s
    """
    A = []
    b = []
    for (xs, ys), (xd, yd) in zip(points_s, points_d):
        A.append([xs, ys, 1, 0, 0, 0])
        A.append([0, 0, 0, xs, ys, 1])
        b.extend([xd, yd])
    vec = np.linalg.solve(np.array(A, dtype=np.float64), np.array(b, dtype=np.float64))
    M = np.array([
        [vec[0], vec[1], vec[2]],
        [vec[3], vec[4], vec[5]],
        [0,      0,      1]
    ], dtype=np.float64)
    return M

### Ma trận biến đổi  Projective

In [ ]:
def homography_matrix(points_s, points_d):
    """
    Tính ma trận Homography 3x3 từ 4 cặp điểm bằng giải hệ bằng SVD
    """
    A = []
    for (xs, ys), (xd, yd) in zip(points_s, points_d):
        A.append([-xs, -ys, -1, 0, 0, 0, xd * xs, xd * ys, xd])
        A.append([0, 0, 0, -xs, -ys, -1, yd * xs, yd * ys, yd])
    A = np.array(A, dtype=np.float64)

    # Giải hệ A * h = 0 bằng SVD
    _, _, Vh = np.linalg.svd(A)
    H = Vh[-1].reshape(3, 3)
    return H / H[2, 2]

### Thực hiện Translation

In [ ]:
h, w = img.shape[:2]
# Tịnh tiến (dx=50, dy=30)
dx, dy = 50, 30
M_trans = np.array([
    [1.0, 0.0, dx],
    [0.0, 1.0, dy],
    [0.0, 0.0, 1.0]
])
img_translated = warp_perspective(img, M_trans, (h, w))

titles = [
    'Original Image',
    'Translation (+50x, +30y)'
]
images = [img, img_translated]

num_images = len(images)
rows = 1
cols = num_images

plt.figure(figsize=(9, 7))
for i in range(num_images):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(titles[i], fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

### Thực hiện Rotation

In [ ]:
# Xoay 45 độ quanh tâm ảnh
theta = np.radians(45)
cx, cy = w / 2.0, h / 2.0

T_to_origin = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]])
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0,              0,             1]])
T_back = np.array([[1, 0, cx], [0, 1, cy], [0, 0, 1]])
M_rot = T_back @ R @ T_to_origin
img_rotated = warp_perspective(img, M_rot, (h, w))

titles = [
    'Original Image',
    'Rotation (45 degree)'

]
images = [img, img_rotated]

num_images = len(images)
rows = 1
cols = num_images

plt.figure(figsize=(9, 7))
for i in range(num_images):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(titles[i], fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

### Thực hiện Scaling

In [ ]:
# Scaling (sx=0.6, sy=0.6)
sx, sy = 0.6, 0.6
M_scale = np.array([
    [sx, 0.0, 0.0],
    [0.0, sy, 0.0],
    [0.0, 0.0, 1.0]
])
img_scaled = warp_perspective(img, M_scale, (h, w))

titles = [
    'Original Image',
    'Scaling (0.6x)'

]
images = [img, img_scaled]

num_images = len(images)
rows = 1
cols = num_images

plt.figure(figsize=(9, 7))
for i in range(num_images):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(titles[i], fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

### Thực hiện biến đổi Affine

In [ ]:
# Xác định qua 3 cặp điểm
pts_src_affine = np.float64([[0, 0], [w - 1, 0], [0, h - 1]])
pts_dst_affine = np.float64([[w * 0.1, h * 0.2], [w * 0.85, h * 0.1], [w * 0.2, h * 0.9]])
M_affine = affine_matrix(pts_src_affine, pts_dst_affine)
img_affine = warp_perspective(img, M_affine, (h, w))

titles = [
    'Original Image',
    'Affine Transform (3 pts)'

]
images = [img, img_affine]

num_images = len(images)
rows = 1
cols = num_images

plt.figure(figsize=(9, 7))
for i in range(num_images):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(titles[i], fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

### Thực hiện biến đổi Projective

In [ ]:
# Xác định qua 4 cặp điểm
pts_src_proj = np.float64([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
pts_dst_proj = np.float64([[w * 0.25, h * 0.1], [w * 0.75, h * 0.1], [w - 1, h * 0.9], [0, h * 0.9]])
M_proj = homography_matrix(pts_src_proj, pts_dst_proj)
img_projective = warp_perspective(img, M_proj, (h, w))

titles = [
    'Original Image',
    'Projective Transform (4 pts)'

]
images = [img, img_projective]

num_images = len(images)
rows = 1
cols = num_images

plt.figure(figsize=(9, 7))
for i in range(num_images):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(titles[i], fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

### So sánh kết quả biến đổi Affine và Projective

In [ ]:
titles = [
    'Affine Transform (3 pts)',
    'Projective Transform (4 pts)'

]
images = [img_affine, img_projective]

num_images = len(images)
rows = 1
cols = num_images

plt.figure(figsize=(9, 7))
for i in range(num_images):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(titles[i], fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()